# DCT Laboratory — Volume II, Chapter 1
## Introduction to Enterprise Optimization
**Seed `26201`** · Companion to Chapter 1 and **AXIOM Lab 2.1** (module **AXIOM-01**) · Mirrored in `DCT_V2_Ch01_Lab.xlsx`

This laboratory reproduces **Worked Example 1.1, *Can Meridian Sign Volume I's Answer?***, with the book's
declarations, and works the five steps of AXIOM Lab 2.1: formulate, run the gates, explore the decision
space, rescale then reweight, add a commitment. It ends with the two-channel budget of Online Appendix
1.B.2. Every Meridian number below is the book's (Table 1.3, Figure 1.4, Online Appendix 1.B.4); the final
cell checks them. Money is in \$ million.

## The declarations (Worked Example 1.1, Step 1)

Meridian arrives with Volume I's declarations: eight readings, from liquidity of \$620 million to a
strategic-risk index of 64, and a three-year digital program whose planned liquidity dips to \$482 million
in its seventh quarter. Volume I's Chapter 8 model adds two kinds of uncertainty: execution and market
deviations, $E' = 0.6E + w$ with $w$ of standard deviation \$50 million a quarter, and closures of the
funding window, 0.10 a year, each removing a gamma-distributed amount (mean \$120 million, standard
deviation \$50 million) that is never recovered. The board's commitment: at most a 5 percent chance that
liquidity falls below the \$350 million floor in some quarter of the program.

Two levers, each with an owner and a price. The **standby line** $l$ (treasury) replaces up to $l$ of each
closure; its fees, about 1 percent a year for three years, cost $0.03\,l$. The **delay** $\phi$ (the
transformation office) phases the build, cutting the trough of the plan from 28 to $28 - 4\phi$ index
points; each quarter costs \$8 million of Digital Solutions' contribution. The box is $0 \le l \le 400$,
$0 \le \phi \le 6$. Breach odds are counted on the book's 100,000 simulated paths, drawn exactly as
Volume I draws them (seed 8, Online Appendix 1.B.4); the lab's seed draws fresh paths in Panel 7.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26201        # the lab's seed: fresh paths for Panel 7
BOOK_SEED = 8       # Volume I's Chapter 8 stream, on which the book counts every breach (Online Appendix 1.B.4)

STATE = {"liquidity ($m)": 620, "net leverage (x EBITDA)": 3.4, "workforce capability": 58, "technology platform": 41,
         "operational efficiency": 71, "ROIC (%)": 8.3, "strategic risk": 64, "market share (%)": 19}

# The digital program's liquidity ($ million), quarterly over three years: Volume I's Chapter 8 model
N, Q = 100_000, 12
LIQ0, FLOOR = 620.0, 350.0                # liquidity today; the board's floor
A_E, SD_W = 0.6, 50.0                     # execution and market: E' = 0.6 E + w, w with sd $50m a quarter
LAM, J_MEAN, J_SD = 0.10, 120.0, 50.0     # funding-window closures: 0.10 a year, gamma sizes, never recovered
DEPTH0, DEPTH_PER_Q = 28.0, 4.0           # the plan's trough (index points); a quarter of delay: 4 points shallower
FEE, VQ, APPETITE = 0.03, 8.0, 0.05       # line fees over the program; a quarter of delay ($m); the appetite
L_MAX, PHI_MAX, TRANCHE = 400.0, 6.0, 50.0

def relax(x0, x_inf, k, t):
    return x_inf + (x0 - x_inf) * np.exp(-k * t)

def trough(depth, tau, t):
    return depth * (t / tau) * np.exp(1 - t / tau)

def plan(depth, t):                       # planned liquidity ($m); index 100 = $620 million today
    return 6.2 * (relax(100, 112, 0.35, t) - trough(depth, 2.0, t))

TQ = np.arange(1, Q + 1) / 4              # the twelve quarters, in years
BASE = plan(0.0, TQ)                      # the plan without its trough
SHAPE = 6.2 * trough(1.0, 2.0, TQ)        # the trough, per index point of depth

def draw_paths(seed, n=N):
    # Volume I's random stream in the book's order: each quarter the execution shocks,
    # the closure counts, then the closure sizes path by path
    rng = np.random.default_rng(seed)
    E, e, closures = np.empty((n, Q)), np.zeros(n), []
    for k in range(Q):
        w = rng.normal(0.0, SD_W, n)
        count = rng.poisson(LAM / 4, n)
        size = rng.gamma((J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN, count.sum())
        closures.append((np.repeat(np.arange(n), count), size))
        e = A_E * e + w
        E[:, k] = e
    return E, closures

class Paths:
    # A line l replaces up to l of each closure; phi quarters of delay make the trough 28 - 4 phi deep.
    # A path breaches the floor exactly when the trough is deeper than its critical depth,
    # min over quarters of (liquidity without the trough - floor) / (trough per point of depth),
    # so the odds of every package are exact counts on the same paths.
    def __init__(self, seed):
        self.E, self.closures = draw_paths(seed)
        self.memo = {}
    def critical(self, line):
        key = round(float(line), 6)
        if key not in self.memo:
            lost, cum = np.empty((N, Q)), np.zeros(N)
            for k, (path, size) in enumerate(self.closures):
                cum += np.bincount(path, np.maximum(size - line, 0.0), N)
                lost[:, k] = cum
            self.memo[key] = np.sort(((BASE + self.E - lost - FLOOR) / SHAPE).min(axis=1))
        return self.memo[key]
    def odds(self, line, phi):
        return np.searchsorted(self.critical(line), DEPTH0 - DEPTH_PER_Q * phi) / N
    def delay_for(self, line, p=APPETITE):
        # least delay meeting p: the depth it allows is a quantile of the critical depths
        d = min(DEPTH0, self.critical(line)[int(np.floor(p * N))])
        return np.nan if d < 0 else (DEPTH0 - d) / DEPTH_PER_Q

def cost(line, phi, v=VQ):
    return FEE * line + v * phi

B = Paths(BOOK_SEED)
print("Meridian's state:", "; ".join(f"{k} {v}" for k, v in STATE.items()))
path0 = plan(DEPTH0, np.arange(Q + 1) / 4)
print("planned liquidity by quarter ($m):", " ".join(f"{x:.0f}" for x in path0),
      f"-> low point {path0.min():.0f} in quarter {path0.argmin()}, {path0.min() - FLOOR:.0f} above the floor")
print(f"the book's paths give Volume I's odds: no lever {100 * B.odds(0, 0):.1f}%, a $100m line {100 * B.odds(100, 0):.1f}%, "
      f"two quarters of delay {100 * B.odds(0, 2):.1f}%, both {100 * B.odds(100, 2):.1f}%")

## Panel 1 — Formulate: the canonical form (Worked Example 1.1, Step 1; Theorem 1.1, Proposition 1.1)

AXIOM Lab 2.1, step 1. The formulation assistant takes the two levers with their owners and prices, the
cost objective and the board's appetite, and emits the canonical form (Equation 1.1): the cost
$0.03\,l + 8\,\phi$ with its sign reversed, over the box, subject to the state map and the one commitment.
Delete the declaration of the delay cost, its price and its owner, and the assistant refuses: a term that
cannot be evaluated from the declaration defines no optimization problem (Proposition 1.1), and a solver
that answered anyway would be solving a problem the company never stated.

In [ ]:
DECLARATION = {
    "decisions": [
        {"symbol": "l", "name": "standby line", "unit": "$ million", "box": (0, 400), "owner": "treasury",
         "price": FEE, "basis": "fees of about 1 percent a year for three years"},
        {"symbol": "phi", "name": "delay of the build", "unit": "quarters", "box": (0, 6), "owner": "transformation office",
         "price": VQ, "basis": "Digital Solutions' contribution a quarter"}],
    "state map": "Volume I's Chapter 8 model of the program's liquidity, 100,000 paths",
    "commitments": [{"name": "P(liquidity below 350 in some quarter)", "limit": APPETITE, "owner": "the board"}],
}

def formulate(decl):
    # emit the canonical form, or refuse when a term has no declared price or owner (Proposition 1.1)
    missing = [d["name"] for d in decl["decisions"] if d.get("price") is None or not d.get("owner")]
    missing += [c["name"] for c in decl["commitments"] if not c.get("owner")]
    if missing:
        return ("REFUSED: no declared price or owner for " + ", ".join(missing) +
                ": the objective is not a function of the declaration (Proposition 1.1)")
    cost_terms = " + ".join(f"{d['price']:g} {d['symbol']}" for d in decl["decisions"])
    lines = [f"maximize    -({cost_terms})    the package's cost in $ million, with its sign reversed"]
    for i, d in enumerate(decl["decisions"]):
        lines.append(f"{'over' if i == 0 else '':12s}{d['box'][0]} <= {d['symbol']} <= {d['box'][1]}    "
                     f"({d['unit']}; {d['owner']}; {d['price']:g} a unit: {d['basis']})")
    lines.append(f"subject to  x = f(l, phi): {decl['state map']}")
    lines += [f"            {c['name']} - {c['limit']:g} <= 0    ({c['owner']})" for c in decl["commitments"]]
    return "\n".join(lines)

print(formulate(DECLARATION))
undeclared = {**DECLARATION, "decisions": [DECLARATION["decisions"][0],
                                           {**DECLARATION["decisions"][1], "price": None, "owner": None}]}
print("\nthe delay cost's price and owner deleted:\n" + formulate(undeclared))

## Panel 2 — Run the gates (Worked Example 1.1, Step 2; Theorem 1.5: \$138.5 million of \$270 million)

AXIOM Lab 2.1, step 2. The four readiness gates of Theorem 1.5, run on the declaration with Volume I's
package: a \$170 million line and the least delay that meets the appetite with it. (R2) is Volume I's joint
check: the program's deepest draw and the cash buffer for a 90th-percentile closure must fit in the
\$270 million above the floor; with the package they use \$138.5 million. Then remove the standby line from
the decision inventory and rerun (R2): the check fails, and the minimal infeasible set names the
commitments that cannot hold together, each with its owner.

In [ ]:
L_V1 = 170.0
PHI_V1 = B.delay_for(L_V1)                # Volume I's package: $170m and the least delay that meets 5 percent
P90 = stats.gamma.ppf(0.9, (J_MEAN / J_SD) ** 2, scale=J_SD ** 2 / J_MEAN)     # a 90th-percentile closure

def deepest_draw(phi):                    # the plan's deepest draw on the cash above the floor
    return LIQ0 - plan(DEPTH0 - DEPTH_PER_Q * phi, np.arange(Q + 1) / 4).min()

def joint_check(line, phi, commitments=("program draw", "closure buffer", "floor")):
    claims = ((deepest_draw(phi) if "program draw" in commitments else 0.0)
              + (max(P90 - line, 0.0) if "closure buffer" in commitments else 0.0))
    room = LIQ0 - FLOOR if "floor" in commitments else np.inf
    return claims, claims <= room

EBITDA = 0.118 * 4200
NET_DEBT, COVENANT = 3.4 * EBITDA, 4.5
def leverage(line):                       # net debt over EBITDA with the line fully drawn
    return (NET_DEBT + line) / EBITDA

core = np.zeros((4, 4))                   # the feedback core: Industrial Systems, Digital, engineering, treasury
for i, j in ((0, 2), (1, 2), (0, 3), (1, 3)):
    core[i, j] = core[j, i] = 0.40        # four two-way links at the ceiling of 0.40
rho, rule = max(abs(np.linalg.eigvals(core))), 0.5 ** 0.25

claims, ok = joint_check(L_V1, PHI_V1)
print(f"Volume I's package: a ${L_V1:.0f}m line and {PHI_V1:.3f} quarters of delay")
print(f"(R1) declaration completeness: every term has a price and an owner: "
      f"{'pass' if not formulate(DECLARATION).startswith('REFUSED') else 'FAIL'}")
print(f"(R2) unified consistency: draw {deepest_draw(PHI_V1):.1f} + closure buffer {max(P90 - L_V1, 0):.1f} = {claims:.1f} "
      f"of {LIQ0 - FLOOR:.0f}: {'pass' if ok else 'FAIL'}")
print(f"(R3) analytical panel: a closed, bounded box; the feedback core's spectral radius {rho:.2f} <= {rule:.3f}: "
      f"{'pass' if rho <= rule else 'FAIL'}")
print(f"(R4) feasibility over the horizon: odds {100 * B.odds(L_V1, PHI_V1):.2f}% <= 5%; leverage with the line drawn "
      f"{leverage(L_V1):.2f}x <= {COVENANT}x: {'pass' if B.odds(L_V1, PHI_V1) <= APPETITE and leverage(L_V1) <= COVENANT else 'FAIL'}")

claims0, ok0 = joint_check(0.0, PHI_V1)
print(f"\nthe line removed from the inventory: draw {deepest_draw(PHI_V1):.1f} + closure buffer {P90:.1f} = {claims0:.1f} "
      f"on {LIQ0 - FLOOR:.0f}: (R2) {'passes' if ok0 else 'fails'}")
OWNERS = {"program draw": "the transformation office", "closure buffer": "treasury", "floor": "the board"}
named = list(OWNERS)
for c in list(named):                     # deletion filter: drop a commitment while the rest still cannot hold
    rest = [x for x in named if x != c]
    if not joint_check(0.0, PHI_V1, rest)[1]:
        named = rest
print("minimal infeasible set:", "; ".join(f"the {c} ({OWNERS[c]})" for c in named),
      "-- drop any one and the others can hold")
print(f"Volume I's declaration before its levers: {deepest_draw(0):.1f} + {P90:.1f} = {joint_check(0.0, 0.0)[0]:.1f} on 270, "
      f"short {joint_check(0.0, 0.0)[0] - 270:.1f}")

## Panel 3 — Solve with continuous levers (Worked Example 1.1, Step 3; Theorem 1.2: \$170 million and 0.67 quarters, \$10.44 million)

With $l$ and $\phi$ continuous the decision space is a box, closed, bounded and convex, and the breach
odds move continuously with the levers, so a best package exists (Theorem 1.2). For each line on Volume I's
grid of \$5 million, the least delay that meets the appetite is read off the paths; the best package is the
cheapest point of that curve: Volume I's answer, a \$170 million line with 0.67 quarters of delay,
\$10.44 million, with the appetite met exactly. Neither lever works alone: the best line on its own leaves
the odds at 8.4 percent, and delay on its own needs more than five quarters.

In [ ]:
LINES = np.arange(0.0, L_MAX + 0.01, 5.0)     # Volume I's grid of lines
CURVE = np.array([B.delay_for(l) for l in LINES])   # least delay meeting 5 percent, line by line

def continuous_best(v=VQ):
    c = FEE * LINES + v * CURVE
    i = int(np.nanargmin(c))
    return LINES[i], CURVE[i], c[i]

L_C, PHI_C, C_C = continuous_best()
print(f"best package: a ${L_C:.0f}m line and {PHI_C:.3f} quarters of delay ({3 * PHI_C:.1f} months), ${C_C:.2f}m "
      f"(fees {FEE * L_C:.2f} + delay {VQ * PHI_C:.2f}); odds {100 * B.odds(L_C, PHI_C):.2f}%")
flat = LINES[FEE * LINES + VQ * CURVE <= C_C + 0.5]
print(f"a flat optimum: every line from ${flat.min():.0f}m to ${flat.max():.0f}m costs within $0.5m of the best")
print(f"the line alone: odds never below {100 * B.odds(L_MAX, 0):.2f}% (a ${L_MAX:.0f}m line; swings in execution "
      f"and markets breach that often on their own)")
print(f"delay alone: {B.delay_for(0.0):.2f} quarters, ${cost(0, B.delay_for(0.0)):.2f}m")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
for phi, col, lab in ((0.0, "#B71C1C", "no delay: the line alone"), (PHI_C, "#1F4E79", f"{PHI_C:.2f} quarters (Volume I)"),
                      (1.0, "#E65100", "one quarter"), (2.0, "#616161", "two quarters")):
    ax.plot(LINES, [100 * B.odds(l, phi) for l in LINES], color=col, lw=2, label=lab)
ax.axhline(100 * APPETITE, color="#999999", lw=0.8, ls="--")
ax.text(398, 5.5, "the board's appetite, 5%", color="#616161", fontsize=8, ha="right")
ax.plot(L_C, 100 * B.odds(L_C, PHI_C), "o", color="#1F4E79", ms=7)
ax.annotate(rf"\${L_C:.0f}m with {PHI_C:.2f} quarters: \${C_C:.2f}m", xy=(L_C, 5.0), xytext=(205, 11.5), fontsize=8,
            color="#1F4E79", arrowprops=dict(arrowstyle="-", color="#1F4E79", lw=0.7))
ax.set(xlim=(0, 400), ylim=(0, 26), xlabel=r"standby line (\$ million)", ylabel="breach odds (%)",
       title="Neither lever alone meets the appetite; together, at \\$10.44m (seed 26201)")
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

## Panel 4 — Make the levers lumpy (Worked Example 1.1, Step 4; Table 1.3, Figure 1.4; Theorem 1.3)

AXIOM Lab 2.1, step 3. The bank writes lines in tranches of \$50 million (0 to 400) and the program office
plans in whole quarters (0 to 6): $9 \times 7 = 63$ packages, a finite set, so the decision space is no longer
convex (Theorem 1.3(ii)). They are few enough to check every one on the same paths: 46 meet the appetite, and
the cheapest is a \$150 million line with one quarter of delay, \$12.50 million, with breach odds of
4.0 percent. Rounding the continuous answer is not a method: down, (150, 0), breaches at 9.6 percent; up,
(200, 1), costs \$14.00 million. The continuous answer is a floor that no lumpy package can beat
(Theorem 1.3(v)), so the lumps cost \$2.06 million: \$1.19 million for whole quarters and \$0.87 million more
for \$50 million tranches.

In [ ]:
LUMPY_LINES, QUARTERS = np.arange(0.0, L_MAX + 0.01, TRANCHE), np.arange(0, 7)
packages = [(l, int(q), B.odds(l, q), cost(l, q)) for l in LUMPY_LINES for q in QUARTERS]
feasible = [p for p in packages if p[2] <= APPETITE]
L_B, Q_B, P_B, C_B = min(feasible, key=lambda p: (p[3], p[2]))
print(f"{len(LUMPY_LINES)} lines x {len(QUARTERS)} delays = {len(packages)} packages; {len(feasible)} meet the appetite (*)")
print("breach odds (%)    quarters of delay:" + "".join(f"{q:8d}" for q in QUARTERS))
for l in LUMPY_LINES:
    print(("  no line" if l == 0 else f"  a ${l:.0f}m line").ljust(38) + "".join(f"{100 * B.odds(l, q):7.2f}{'*' if B.odds(l, q) <= APPETITE else ' '}"
                                                   for q in QUARTERS))
print(f"cheapest that meets the appetite: a ${L_B:.0f}m line, {Q_B} quarter of delay, ${C_B:.2f}m, odds {100 * P_B:.2f}%")
print(f"convexity: the box is convex, the 63 packages are not (the midpoint of (150, 1) and (200, 1) is no package); "
      f"forty yes-or-no choices would make 2^40 = {2 ** 40:.2e} combinations")

def least_line(q, step):
    # least line on a grid of `step` meeting the appetite with q quarters: the odds fall as the line grows
    grid = np.arange(0.0, L_MAX + step / 2, step)
    if B.odds(grid[-1], q) > APPETITE:
        return None
    lo, hi = -1, len(grid) - 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        lo, hi = (lo, mid) if B.odds(grid[mid], q) <= APPETITE else (mid, hi)
    return float(grid[hi])

by_quarter = {int(q): {s: least_line(q, s) for s in (1.0, 10.0, 50.0)} for q in QUARTERS}
print("\nleast line for each whole number of quarters (Online Appendix 1.B.4)")
print("delay   line ($1m steps)   odds    line ($50m)   odds    cost ($m)")
for q in QUARTERS:
    a, b = by_quarter[q][1.0], by_quarter[q][50.0]
    if a is None:
        print(f"{q:4d}        none            --       none        --        --")
        continue
    print(f"{q:4d} {a:12.0f} {100 * B.odds(a, q):12.2f}% {b:10.0f} {100 * B.odds(b, q):8.2f}% {cost(b, q):10.2f}")

def best_whole_quarters(step):
    return min(((by_quarter[q][step], q) for q in by_quarter if by_quarter[q][step] is not None), key=lambda r: cost(*r))

L1, Q1 = best_whole_quarters(1.0)
L10, Q10 = best_whole_quarters(10.0)
TABLE_13 = [("Continuous levers (Volume I)", L_C, PHI_C), ("Whole quarters, line to the $1 million", L1, Q1),
            ("Whole quarters, $10 million tranches", L10, Q10), ("Whole quarters, $50 million tranches", L_B, Q_B),
            ("Rounded down (fails the appetite)", TRANCHE * np.floor(L_C / TRANCHE), np.floor(PHI_C)),
            ("Rounded up", TRANCHE * np.ceil(L_C / TRANCHE), np.ceil(PHI_C))]
print("\nTable 1.3                                  line ($m)   delay (q)   odds    cost ($m)")
for name, l, phi in TABLE_13:
    delay = f"{phi:.2f}" if phi != round(phi) else f"{phi:.0f}"
    print(f"{name:42s} {l:8.0f} {delay:>10s} {100 * B.odds(l, phi):9.2f}% {cost(l, phi):9.2f}")
lumps = C_B - C_C
print(f"\nthe lumps cost ${lumps:.2f}m ({100 * lumps / C_C:.0f}%): whole quarters ${cost(L1, Q1) - C_C:.2f}m, "
      f"then $50m tranches ${C_B - cost(L1, Q1):.2f}m; $10m tranches would save ${C_B - cost(L10, Q10):.2f}m")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
TOP = 2.45                                # quarters 0 to 2 shown: every best package lies there
ax.set(xlim=(-8, 408), ylim=(-0.42, TOP), xticks=range(0, 401, 50), yticks=[0, 1, 2],
       xlabel=r"standby line (\$ million)", ylabel="delay of the build (quarters)",
       title="The plane of packages: the best lumpy one costs \\$2.06m more (seed 26201)")
ax.fill_between(LINES, np.minimum(CURVE, TOP), TOP, color="#EAF1E8", lw=0)
ax.plot(LINES, CURVE, color="#2E7D32", lw=1.6)
ax.text(104, 2.2, "breach odds = 5%", color="#2E7D32", fontsize=8)
ax.text(330, 2.22, "appetite met", color="#2E7D32", fontsize=8, style="italic", ha="center")
fig.canvas.draw()
p0, p1 = ax.transData.transform((0.0, 0.0)), ax.transData.transform((100.0, -100 * FEE / VQ))
angle = np.degrees(np.arctan2(p1[1] - p0[1], p1[0] - p0[0]))
for c in (C_C, C_B, cost(200, 1)):
    ax.plot(LINES, (c - FEE * LINES) / VQ, color="#616161", lw=0.7, ls=(0, (4, 2)), zorder=1)
    ax.text(24, (c - FEE * 24) / VQ, rf"\${c:.2f}m", color="#616161", fontsize=7.5, ha="center", va="center", rotation=angle,
            rotation_mode="anchor", bbox=dict(boxstyle="square,pad=0.1", fc="white", ec="none"), zorder=2)
for l, q, p, c in packages:
    if q <= 2:
        ax.plot(l, q, "o", ms=4.5, mfc="#1A1A1A" if p <= APPETITE else "white", mec="#1A1A1A", mew=0.8, zorder=3)
for v, dx, dy, ha in ((4.0, -7, -0.13, "right"), (16.0, 7, -0.15, "left")):
    l, phi, _ = continuous_best(v)
    ax.plot(l, phi, "D", ms=5.5, mfc="white", mec="#1F4E79", mew=1.0, zorder=4)
    ax.text(l + dx, phi + dy, rf"\${v:.0f}m a quarter", color="#1F4E79", fontsize=7.5, ha=ha, va="center")
ax.plot(L_C, PHI_C, "o", ms=8, color="#1F4E79", zorder=5)
ax.annotate(rf"continuous best ({L_C:.0f}, {PHI_C:.2f}): \${C_C:.2f}m", xy=(L_C, PHI_C), xytext=(64, 0.42), fontsize=8,
            color="#1F4E79", ha="center", va="center", arrowprops=dict(arrowstyle="-", color="#1F4E79", lw=0.7, shrinkB=4))
ax.plot(L_B, Q_B, "o", ms=13, mfc="none", mec="#5E35B1", mew=1.8, zorder=5)
ax.annotate(rf"best lumpy ({L_B:.0f}, {Q_B}): \${C_B:.2f}m, odds {100 * P_B:.1f}%", xy=(L_B, Q_B), xytext=(300, 1.55),
            fontsize=8, color="#5E35B1", ha="center", va="center",
            arrowprops=dict(arrowstyle="-", color="#5E35B1", lw=0.7, shrinkB=6))
ax.plot(200, 1, "o", ms=13, mfc="none", mec="#E65100", mew=1.4, zorder=5)
ax.annotate(rf"rounded up (200, 1): \${cost(200, 1):.2f}m", xy=(200, 1), xytext=(318, 0.8), fontsize=8, color="#E65100",
            ha="center", va="center", arrowprops=dict(arrowstyle="-", color="#E65100", lw=0.7, shrinkB=6))
ax.plot(150, 0, "x", ms=9, mew=2, color="#B71C1C", zorder=5)
ax.annotate(f"rounded down (150, 0): odds {100 * B.odds(150, 0):.1f}%", xy=(150, 0), xytext=(300, 0.27), fontsize=8,
            color="#B71C1C", ha="center", va="center", bbox=dict(boxstyle="square,pad=0.1", fc="white", ec="none"),
            arrowprops=dict(arrowstyle="-", color="#B71C1C", lw=0.7, shrinkB=5))
ax.text(-4, -0.33, "dots: lumpy packages (filled meet the appetite); dashed: equal cost; diamonds: the continuous best "
        "when delay is reweighted", fontsize=7, color="#616161", va="center")
plt.tight_layout(); plt.show()

## Panel 5 — Rescale, then reweight (Worked Example 1.1, Step 5; Theorem 1.4)

AXIOM Lab 2.1, step 4. Stating every cost after tax, or in thousands of dollars, is a strictly increasing
transformation of the objective and changes no package (Theorem 1.4(iii)). Changing what a quarter of delay
is worth is a reweighting, a new objective (Theorem 1.4(iv)): at \$4 million a quarter the continuous answer
moves to a \$140 million line and 0.83 quarters (\$7.52 million), at \$16 million to a \$195 million line and
0.60 quarters (\$15.51 million). The lumpy answer does not move: for every value of delay above \$1.5 million
a quarter, \$150 million with one quarter stays the cheapest lumpy package, because no lumpy package meets the
appetite without a quarter of delay and none with one quarter needs less than \$150 million.

In [ ]:
order = lambda f: [p[:2] for p in sorted(feasible, key=lambda p: (f(p[3]), p[2]))]
same = order(lambda c: c) == order(lambda c: 0.75 * c) == order(lambda c: 1000 * c)
print(f"after tax at 25 percent, or in thousands of dollars: every one of the {len(feasible)} feasible packages keeps "
      f"its rank: {same}; the cheapest stays ({L_B:.0f}, {Q_B})")

print("\na quarter of delay   continuous best: line   delay    cost     cheapest lumpy: line   delay    cost")
for v in (4.0, 8.0, 16.0):
    l, phi, c = continuous_best(v)
    lb = min(feasible, key=lambda p: (cost(p[0], p[1], v), p[2]))
    print(f"{'$%.0fm' % v:>10s} {l:28.0f} {phi:7.2f} {c:7.2f} {lb[0]:23.0f} {lb[1]:7d} {cost(lb[0], lb[1], v):7.2f}")
moved = continuous_best(16.0)[0] - continuous_best(4.0)[0]
print(f"the board's number moves the continuous line by ${moved:.0f}m; the signed package stays the same")

fee_q = {q: FEE * by_quarter[q][50.0] for q in by_quarter if by_quarter[q][50.0] is not None}
print("\ncheapest lumpy package with q quarters, at v a quarter: " + "; ".join(f"{fee_q[q]:.1f} + {q}v" for q in fee_q))
threshold = max((fee_q[1] - fee_q[q]) / (q - 1) for q in fee_q if q > 1)
print(f"one quarter is the cheapest exactly when v > {threshold:.2f} (two and three quarters tie at that value)")

for v in (8.0, 16.0):
    l, phi, c = continuous_best(v)
    near = (TRANCHE * round(l / TRANCHE), round(phi))
    print(f"at ${v:.0f}m the nearest package to ({l:.0f}, {phi:.2f}) is ({near[0]:.0f}, {near[1]}): ${cost(*near, v):.2f}m, "
          f"against ${cost(L_B, Q_B, v):.2f}m for ({L_B:.0f}, {Q_B})")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
for v, col in ((4.0, "#616161"), (8.0, "#1F4E79"), (16.0, "#E65100")):
    l, phi, c = continuous_best(v)
    ax.plot(LINES, FEE * LINES + v * CURVE, color=col, lw=2, label=rf"\${v:.0f}m a quarter: best line \${l:.0f}m, \${c:.2f}m")
    ax.plot(l, c, "o", color=col, ms=7)
    ax.plot(L_B, cost(L_B, Q_B, v), "s", mfc="white", mec=col, mew=1.4, ms=7)
ax.plot([], [], "s", mfc="white", mec="#1A1A1A", ms=7, label=r"the signed package (150, 1) at each value")
ax.set(xlim=(60, 330), ylim=(4, 26), xlabel=r"standby line (\$ million)",
       ylabel=r"cost on the 5% curve (\$ million)",
       title="Reweighting delay moves the continuous best by \\$55m of line (seed 26201)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Panel 6 — Add a commitment (AXIOM Lab 2.1, step 5; Proposition 1.2)

AXIOM Lab 2.1, step 5. Feasibility is conjunction: a package is feasible only if it meets every commitment at
once, so adding a commitment can only shrink the feasible packages, and the order in which commitments are
listed changes nothing (Proposition 1.2); nor does writing one with another sign convention or scale
(Theorem 1.1(ii)). Add the capability floor of 55 for the digital build and the covenant, net leverage at most
4.5 times EBITDA, with the line fully drawn; then Volume I's joint check, which the signed package still
passes with \$120 million of cash above the floor unclaimed.

In [ ]:
CAPABILITY, CAPABILITY_FLOOR = 58.0, 55.0    # capability stays at 58 through the digital program
COMMITMENTS = {
    "the board's appetite": lambda l, q, p: p <= APPETITE,
    "the covenant, with the line fully drawn": lambda l, q, p: leverage(l) <= COVENANT,
    "the capability floor of 55": lambda l, q, p: CAPABILITY >= CAPABILITY_FLOOR,
    "Volume I's joint check": lambda l, q, p: joint_check(l, q)[1],
}
def meets(names):
    return [(l, q) for l, q, p, c in packages if all(COMMITMENTS[n](l, q, p) for n in names)]

print("commitments in force                               feasible packages")
names = []
for n in COMMITMENTS:
    names.append(n)
    print(f"  {'+ ' if len(names) > 1 else '  '}{n:46s} {len(meets(names)):6d}")
print("listed in the opposite order, the same packages:", meets(names[::-1]) == meets(names))
rewritten = [(l, q) for l, q, p, c in packages if 100 * (APPETITE - p) >= 0]
print("the appetite written as 100 (0.05 - P) >= 0, the same packages:", rewritten == meets(names[:1]))
print(f"none drops out: the covenant would bind only for a line above ${COVENANT * EBITDA - NET_DEBT:.0f}m, and "
      f"capability stays at {CAPABILITY:.0f}")

claims_b, _ = joint_check(L_B, Q_B)
print(f"\nthe signed package ({L_B:.0f}, {Q_B}), commitment by commitment:")
print(f"  appetite: odds {100 * P_B:.2f}% against 5%, a margin of {100 * (APPETITE - P_B):.2f} points")
print(f"  covenant: {leverage(L_B):.2f}x against {COVENANT}x with the line fully drawn")
print(f"  capability: {CAPABILITY:.0f} against the floor of {CAPABILITY_FLOOR:.0f}")
print(f"  joint check: draw {deepest_draw(Q_B):.1f} + closure buffer {P90 - L_B:.1f} = {claims_b:.1f} of 270: "
      f"${270 - claims_b:.0f}m of cash above the floor unclaimed")

## Panel 7 — Route the problem and certify the answer (Worked Example 1.1, Step 6; Definition 1.7; Online Appendix 1.B.4)

The levers are set once, at the start, and the commitment is a probability: a static problem under
uncertainty, solved in Chapter 9 with chance constraints and sample averages, and lumpy, which brings in the
integer methods of Chapter 4. A rule (draw on the line only after a funding window closes, slow the build when
cash falls) would make it a problem of policies (Part II). An answer is worth its certificates
(Definition 1.7): the residual of every commitment (Panel 6), a bound on how much better any package could be,
and, because the odds are estimated on a sample, the sample's error. Each estimate on $N = 100{,}000$ paths has
standard error $\sqrt{\hat p(1 - \hat p)/N}$: 0.07 percentage points near 5 percent, 0.06 near 4 percent.

In [ ]:
def se(p):
    return np.sqrt(p * (1 - p) / N)

print(f"standard errors: {100 * se(APPETITE):.3f} points near 5%, {100 * se(P_B):.3f} near 4%")
print(f"the signed package clears the appetite by {100 * (APPETITE - P_B):.2f} points, {(APPETITE - P_B) / se(P_B):.1f} standard errors; "
      f"rounding down misses it by {100 * (B.odds(150, 0) - APPETITE):.2f} points")
print(f"bound: no lumpy package can cost less than the continuous ${C_C:.2f}m (Theorem 1.3(v)); the signed package is "
      f"within ${C_B - C_C:.2f}m of it, and the best of the {len(packages)} by enumeration")

F = Paths(SEED)                           # fresh paths: the same model, the lab's seed
SHOW = [("continuous best", L_C, PHI_C), ("$121m, 1 quarter", L1, Q1), ("$130m, 1 quarter", L10, Q10),
        ("$150m, 1 quarter", L_B, Q_B), ("$200m, 1 quarter", 200.0, 1), ("$150m, no delay", 150.0, 0)]
print(f"\npackage              book's paths    fresh paths (seed {SEED})    fresh against 5%, in standard errors")
for name, l, phi in SHOW:
    pb, pf = B.odds(l, phi), F.odds(l, phi)
    print(f"{name:20s} {100 * pb:10.2f}% {100 * pf:16.2f}% {(pf - APPETITE) / se(pf):24.1f}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
x = np.arange(len(SHOW))
for dx, P, col, lab in ((-0.12, B, "#1F4E79", "the book's paths (seed 8)"), (0.12, F, "#E65100", f"fresh paths (seed {SEED})")):
    p = np.array([P.odds(l, phi) for _, l, phi in SHOW])
    ax.errorbar(x + dx, 100 * p, yerr=196 * se(p), fmt="o", color=col, ms=5, capsize=3, lw=1.2, label=lab)
ax.axhline(100 * APPETITE, color="#B71C1C", lw=0.9, ls="--")
ax.text(len(SHOW) - 0.5, 5.25, "appetite 5%", color="#B71C1C", fontsize=8, ha="right")
ax.set_xticks(x)
ax.set_xticklabels(["continuous best\n(170, 0.67)", "\\$121m\n1 quarter", "\\$130m\n1 quarter", "\\$150m\n1 quarter",
                    "\\$200m\n1 quarter", "\\$150m\nno delay"], fontsize=8)
ax.set(ylim=(2.5, 10.5), ylabel="breach odds (%), with 95% intervals",
       title="Fresh paths move the boundary packages, not the lumpy answer (seed 26201)")
ax.legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

On the book's paths the packages on the 5 percent boundary meet the appetite by construction. On fresh paths
their odds move by sampling error: with the lab's seed the \$121 million line gives 5.04 percent, within a
standard error of 5, and the continuous optimum 5.14 percent, two standard errors above (a package chosen on
the paths that certify it tends to look a little better on them than on fresh ones). The lumpy answer does not
hang on the draw: on both sets of paths it clears the appetite by about a point, and rounding down misses it by
more than four and a half.

## Panel 8 — A two-channel budget and its price (Online Appendix Example 1.B.2: $x^* = (36, 64)$, value 50, multiplier 0.25)

Online Appendix 1.B.2 sends one more case to this notebook, a small one with a price you can see, not about
Meridian: allocate a budget of 100 between two channels with returns $3\sqrt{x_1}$ and $4\sqrt{x_2}$. The
objective is concave and the region convex and compact, so a maximum exists and every local maximum is global.
At the optimum the budget binds and marginal returns are equal, $3/(2\sqrt{x_1}) = 4/(2\sqrt{x_2}) = \lambda$:
$x^* = (36, 64)$, value 50, multiplier $\lambda = 0.25$. The multiplier is a price: for a budget $B$ the value
is $5\sqrt{B}$, whose slope at 100 is 0.25. Equal marginal returns and the multiplier together are the
certificate: anyone can check them without solving the problem again.

In [ ]:
a1, a2, budget = 3.0, 4.0, 100.0
x1 = budget * a1 ** 2 / (a1 ** 2 + a2 ** 2)
x2 = budget - x1
value = a1 * np.sqrt(x1) + a2 * np.sqrt(x2)
lam1, lam2 = a1 / (2 * np.sqrt(x1)), a2 / (2 * np.sqrt(x2))
print(f"x* = ({x1:.0f}, {x2:.0f}), value {value:.2f}; marginal returns {lam1:.4f} and {lam2:.4f}: equal, the multiplier")
grid = np.linspace(0.0, budget, 100_001)
f = a1 * np.sqrt(grid) + a2 * np.sqrt(budget - grid)
print(f"a search along the budget line agrees: best x1 = {grid[f.argmax()]:.2f}, value {f.max():.4f}")
V = lambda b: np.sqrt(a1 ** 2 + a2 ** 2) * np.sqrt(b)      # the value of a budget b: 5 sqrt(b)
print(f"the value of the budget, 5 sqrt(B): V(100) = {V(budget):.2f}; slope at 100 = {(V(100.01) - V(99.99)) / 0.02:.4f}; "
      f"one more unit is worth {V(101) - V(100):.4f}")

## Validation — the book's numbers, and agreement with `DCT_V2_Ch01_Lab.xlsx`

Each checkpoint is a number printed in Chapter 1 (Section 1.1, Worked Example 1.1, Table 1.3, Figure 1.4) or in
Online Appendix 1.B.2 and 1.B.4. The workbook recomputes the ones marked *workbook* with Excel formulas and
shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
rd, ru = TABLE_13[4], TABLE_13[5]
checks = [
    ("continuous best: line, $m (workbook)", L_C, 170, 0),
    ("continuous best: delay, quarters (workbook)", PHI_C, 0.67, 0.005),
    ("continuous best: cost, $m (workbook)", C_C, 10.44, 0.005),
    ("continuous best: breach odds, %", 100 * B.odds(L_C, PHI_C), 5.00, 0.005),
    ("the line alone: lowest odds, %", 100 * B.odds(L_MAX, 0), 8.40, 0.005),
    ("delay alone: quarters (workbook)", B.delay_for(0.0), 5.09, 0.005),
    ("(R2) joint check with the package, $m (workbook)", claims, 138.5, 0.05),
    ("spectral radius of the feedback core (workbook)", rho, 0.80, 5e-4),
    ("lumpy packages (workbook)", len(packages), 63, 0),
    ("lumpy packages meeting the appetite (workbook)", len(feasible), 46, 0),
    ("best lumpy package: line, $m (workbook)", L_B, 150, 0),
    ("best lumpy package: delay, quarters (workbook)", Q_B, 1, 0),
    ("best lumpy package: cost, $m (workbook)", C_B, 12.50, 0.005),
    ("best lumpy package: breach odds, %", 100 * P_B, 4.00, 0.005),
    ("rounded down (150, 0): breach odds, %", 100 * B.odds(rd[1], rd[2]), 9.59, 0.005),
    ("rounded up (200, 1): breach odds, %", 100 * B.odds(ru[1], ru[2]), 3.39, 0.005),
    ("rounded up (200, 1): cost, $m (workbook)", cost(ru[1], ru[2]), 14.00, 0.005),
    ("whole quarters, $1m steps: line, $m", L1, 121, 0),
    ("whole quarters, $1m steps: breach odds, %", 100 * B.odds(L1, Q1), 4.97, 0.005),
    ("whole quarters, $1m steps: cost, $m (workbook)", cost(L1, Q1), 11.63, 0.005),
    ("$10m tranches: line, $m", L10, 130, 0),
    ("$10m tranches: breach odds, %", 100 * B.odds(L10, Q10), 4.6, 0.05),
    ("$10m tranches: cost, $m (workbook)", cost(L10, Q10), 11.90, 0.005),
    ("the lumps cost, $m (workbook)", lumps, 2.06, 0.005),
    ("of which whole quarters, $m (workbook)", cost(L1, Q1) - C_C, 1.19, 0.005),
    ("$10m tranches would save, $m (workbook)", C_B - cost(L10, Q10), 0.60, 0.005),
    ("two quarters: least line in $1m steps, $m", by_quarter[2][1.0], 70, 0),
    ("two quarters: $50m package cost, $m (workbook)", cost(by_quarter[2][50.0], 2), 19.00, 0.005),
    ("six quarters, no line: breach odds, %", 100 * B.odds(0.0, 6), 3.67, 0.005),
    ("$4m a quarter: continuous line, $m (workbook)", continuous_best(4.0)[0], 140, 0),
    ("$4m a quarter: continuous cost, $m (workbook)", continuous_best(4.0)[2], 7.52, 0.005),
    ("$16m a quarter: continuous line, $m (workbook)", continuous_best(16.0)[0], 195, 0),
    ("$16m a quarter: continuous delay, quarters (workbook)", continuous_best(16.0)[1], 0.60, 0.005),
    ("$16m a quarter: continuous cost, $m (workbook)", continuous_best(16.0)[2], 15.51, 0.005),
    ("lumpy answer holds above, $m a quarter (workbook)", threshold, 1.5, 1e-9),
    ("$16m a quarter: nearest package (200, 1), $m (workbook)", cost(200, 1, 16.0), 22.00, 0.005),
    ("$16m a quarter: (150, 1), $m (workbook)", cost(L_B, Q_B, 16.0), 20.50, 0.005),
    ("cash above the floor unclaimed, $m (workbook)", 270 - claims_b, 120, 0.5),
    ("standard error near 5%, points (workbook)", 100 * se(APPETITE), 0.07, 0.005),
    ("standard error near 4%, points (workbook)", 100 * se(P_B), 0.06, 0.005),
    ("lumpy margin, standard errors (workbook)", (APPETITE - P_B) / se(P_B), 16, 0.5),
    ("rounding down misses by, points (workbook)", 100 * (B.odds(150, 0) - APPETITE), 4.6, 0.05),
    ("two-channel budget: x1* (workbook)", x1, 36, 1e-9),
    ("two-channel budget: value (workbook)", value, 50, 1e-9),
    ("two-channel budget: multiplier (workbook)", lam1, 0.25, 1e-9),
]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"
    ok += status == "PASS"
    print(f"{name:55s} {got:11.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 2 turns the canonical form into an instance in five blocks and prices the way a commitment
holds under uncertainty, on the average path, in a stress set, with stated probability or on every path, for
the same two levers. AXIOM-01 runs this lab interactively (Lab 2.1); this notebook and the workbook are its
reproducible record.